# 01 — Factor-region matrices and iRF inputs

**Objective.** Show how ChIP-informed binary factor occupancy is constructed, then use frozen Blood-derived matrices to create the 13 manuscript-matched PBMC iRF inputs.

**Inputs.** `fileList.tab`, `scplusmdata.h5mu`, direct eRegulon metadata, and frozen HVR/triplet factor matrices. **Outputs.** Grouped Parquet inputs under `WORK_ROOT/01_irf_inputs`.

## Configuration and imports

In [ ]:
import gc
import os
from pathlib import Path

import mudata
import pandas as pd

from _tutorial_utils import ensure_crane_grn_importable, require_path

ensure_crane_grn_importable()
from crane_grn.chip_atlas import (
    align_factor_columns,
    build_filtered_modifier_tracks,
    build_region_by_factor,
    filter_sparse_factors,
    query_chip_atlas,
    regions_to_bed,
    select_available_factors,
)
from crane_grn.io import get_modality
from crane_grn.irf.inputs import build_irf_input, build_irf_inputs_by_group
from crane_grn.regions import canonicalize_regions

DATA_ROOT = Path(os.environ.get("CRANE_GRN_DATA_ROOT", "../../CRANE_GRN_data")).expanduser().resolve()
WORK_ROOT = Path(os.environ.get("CRANE_GRN_TUTORIAL_WORK", "./results")).expanduser().resolve()
PBMC_INPUT_ROOT = DATA_ROOT / "PBMC" / "prepared_inputs"
PBMC_REFERENCE_ROOT = DATA_ROOT / "PBMC" / "outputs"
IRF_INPUT_ROOT = WORK_ROOT / "01_irf_inputs"
IRF_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_LIVE_CHIP_ATLAS = True #If True, will query Chip-Atlas for the factors. If False, will use the pre-downloaded factors in PBMC_INPUT_ROOT.

## A. How the factor-region matrices are constructed

The live block is disabled by default because ChIP-Atlas is network-dependent and its assembled tracks evolve. Candidate factors are first restricted to genes present in the PBMC RNA feature axis. ChIP-Atlas is then queried explicitly with `cell_types="Blood"`; the generic source default remains `All cell types`.

Histone-modifier tracks require biological filtering before they can be interpreted as factor occupancy. Acetyltransferase tracks are restricted to peaks overlapping active acetylation marks, whereas demethylase tracks exclude peaks overlapping the repressive H3K27me3 mark. The PBMC mapping and modes are dataset configuration passed explicitly to the generic source API. The HVR universe itself is an upstream cisTopic-derived input; the live block reconstructs ChIP occupancy on the frozen manuscript HVR coordinates.

In [ ]:
FACTOR_CANDIDATES = [
    "FOXA1", "FOXA2", "GATA3", "GATA4", "SOX2", "KLF4",
    "EBF1", "BATF", "CEBPA", "CEBPB", "SPI1", "RUNX1",
    "IRF8", "TCF7", "LEF1", "BCL11B", "SATB1", "IRF4",
    "PAX5", "SMARCA4", "SMARCA2", "ARID1A", "ARID1B",
    "ARID2", "PBRM1", "SMARCB1", "CHD1", "CHD2", "CHD7",
    "CHD8", "SMARCA5", "SMARCA1", "INO80", "SRCAP", "EP400",
    "EP300", "CREBBP", "KAT2A", "KAT2B", "KDM6A", "KDM6B",
]
HISTONE_MARKS = ["H3K27ac", "H3K27me3", "H3K9ac"]
PBMC_MODIFIER_MARKERS = {
    "EP300": ["H3K27ac", "H3K9ac"],
    "CREBBP": ["H3K27ac", "H3K9ac"],
    "KAT2A": ["H3K27ac", "H3K9ac"],
    "KAT2B": ["H3K27ac", "H3K9ac"],
    "KDM6A": ["H3K27me3"],
    "KDM6B": ["H3K27me3"],
}
PBMC_MODIFIER_MODES = {
    "EP300": "overlap", "CREBBP": "overlap",
    "KAT2A": "overlap", "KAT2B": "overlap",
    "KDM6A": "exclude", "KDM6B": "exclude",
}

def resolve_unique_tracks(downloaded, targets, require_all=False):
    tracks = {}
    missing = []

    for target in targets:
        paths = downloaded.loc[
            downloaded["track_type"] == target,
            "local_path",
        ].tolist()

        if len(paths) == 0:
            missing.append(target)
            continue

        if len(paths) > 1:
            raise ValueError(
                "Expected at most one Blood track for {}; found {}.".format(
                    target, len(paths)
                )
            )

        tracks[target] = paths[0]

    if require_all and missing:
        raise ValueError(
            "Required Blood tracks are missing: {}".format(missing)
        )

    return tracks, missing

In [ ]:
if RUN_LIVE_CHIP_ATLAS:
    live_root = WORK_ROOT / "01_live_chip_atlas"
    live_root.mkdir(parents=True, exist_ok=True)
    scplus_live = mudata.read(str(require_path(PBMC_INPUT_ROOT / "scplusmdata.h5mu", "PBMC scplusmdata input")))
    rna_live = get_modality(scplus_live, "scRNA_counts")
    available_factors, missing_factors = select_available_factors(FACTOR_CANDIDATES, rna_live.var_names)
    downloaded = query_chip_atlas(
        filelist_path=require_path(PBMC_INPUT_ROOT / "fileList.tab", "ChIP-Atlas file list"),
        output_dir=live_root / "tracks",
        factors=available_factors,
        histone_marks=HISTONE_MARKS,
        genome="hg38",
        cell_types="Blood",
        threshold="10",
    )
    factor_tracks, missing_chip_factors = resolve_unique_tracks(
        downloaded,
        available_factors,
        require_all=False,
    )

    histone_tracks, missing_histone_marks = resolve_unique_tracks(
        downloaded,
        HISTONE_MARKS,
        require_all=True,
    )

    print("RNA-present but Blood ChIP-Atlas-absent factors:", missing_chip_factors)
    filtered_modifiers = build_filtered_modifier_tracks(
        factor_tracks=factor_tracks,
        histone_tracks=histone_tracks,
        modifier_to_markers=PBMC_MODIFIER_MARKERS,
        modifier_to_mode=PBMC_MODIFIER_MODES,
    )
    factor_tracks.update(filtered_modifiers)

    frozen_hvr_axis = pd.read_parquet(PBMC_INPUT_ROOT / "region_by_factor_hvr.parquet").index
    hvr_bed = regions_to_bed(frozen_hvr_axis)
    hvr_bed_path = live_root / "hvr_regions.bed"
    hvr_bed.to_csv(hvr_bed_path, sep="\t", header=False, index=False)
    live_hvr = build_region_by_factor(factor_tracks, hvr_bed_path)
    live_hvr, kept_factors, dropped_factors, factor_statistics = filter_sparse_factors(
        live_hvr, min_fraction=0.01
    )

    direct_metadata = pd.read_csv(PBMC_REFERENCE_ROOT / "eRegulon_direct.tsv", sep="\t")
    triplet_bed = regions_to_bed(direct_metadata["Region"].drop_duplicates())
    triplet_bed_path = live_root / "triplet_regions.bed"
    triplet_bed.to_csv(triplet_bed_path, sep="\t", header=False, index=False)
    live_triplet = build_region_by_factor(factor_tracks, triplet_bed_path)
    live_triplet = align_factor_columns(live_triplet, kept_factors)
    live_hvr.to_parquet(live_root / "region_by_factor_hvr_live.parquet")
    live_triplet.to_parquet(live_root / "region_by_factor_triplet_live.parquet")
    print("Live HVR matrix:", live_hvr.shape)
    print("Live triplet matrix:", live_triplet.shape)
    print("RNA-absent candidates:", missing_factors)
else:
    print("Live ChIP-Atlas reconstruction is disabled; continuing with frozen Blood matrices.")

## B. Frozen manuscript matrices

> The live ChIP-Atlas reconstruction demonstrates how the matrices are generated. The frozen matrices distributed with the CRANE-GRN data archive are used below for manuscript-matched execution because ChIP-Atlas is updated over time.

In [ ]:
hvr_factors = pd.read_parquet(require_path(PBMC_INPUT_ROOT / "region_by_factor_hvr.parquet", "frozen HVR factor matrix"))
triplet_factors = pd.read_parquet(require_path(PBMC_INPUT_ROOT / "region_by_factor_triplet.parquet", "frozen triplet factor matrix"))
hvr_factors.index = canonicalize_regions(hvr_factors.index)
triplet_factors.index = canonicalize_regions(triplet_factors.index)
if hvr_factors.shape != (38659, 23):
    raise ValueError("Frozen PBMC HVR factor matrix must have shape (38659, 23).")
if triplet_factors.shape != (11921, 23):
    raise ValueError("Frozen PBMC triplet factor matrix must have shape (11921, 23).")
if list(hvr_factors.columns.astype(str)) != list(triplet_factors.columns.astype(str)):
    raise ValueError("Frozen HVR and triplet factor axes differ.")
print("HVR factor matrix:", hvr_factors.shape)
print("Triplet factor matrix:", triplet_factors.shape)
print("Factors:", hvr_factors.columns.astype(str).tolist())

## C. Build cell-type-specific iRF inputs

For each PBMC cell type, observed accessibility is summarized over the frozen HVR universe. Entropy filtering and the accessibility label are computed independently per group. Only informative, factor-occupied regions are retained.

In [ ]:
scplus = mudata.read(str(require_path(PBMC_INPUT_ROOT / "scplusmdata.h5mu", "PBMC scplusmdata input")))
atac = get_modality(scplus, "scATAC_counts")
if "cell_type" not in atac.obs:
    raise KeyError("Expected group column 'cell_type' was not found in scATAC_counts.obs.")
atac_regions = canonicalize_regions(atac.var_names)
positions = atac_regions.get_indexer(hvr_factors.index)
if (positions < 0).any():
    raise ValueError("Some frozen HVR regions are absent from scATAC_counts.")
hvr_accessibility = atac.X[:, positions]
groups = atac.obs["cell_type"].astype(str).copy()

irf_inputs, irf_summary = build_irf_inputs_by_group(
    accessibility=hvr_accessibility,
    cell_names=atac.obs_names,
    region_names=hvr_factors.index,
    groups=groups,
    region_by_factor=hvr_factors,
    output_dir=IRF_INPUT_ROOT,
    open_fraction_threshold=0.5,
    entropy_minimum=0.05,
    minimum_cells=100,
    minimum_class_regions=100,
    filename_prefix="PBMC_Blood",
    save_bed=True,
)
irf_summary.to_csv(IRF_INPUT_ROOT / "irf_input_summary.tsv", sep="\t", index=False)
if len(irf_inputs) != 13:
    raise ValueError("Expected 13 PBMC iRF inputs; generated {}.".format(len(irf_inputs)))
display(irf_summary)

### One-group equivalent

A one-group dataset does not require the grouped wrapper. The same scientific transformation is applied once with `build_irf_input(accessibility, region_names, region_by_factor, open_fraction_threshold=0.5, entropy_minimum=0.05)`. The disabled cell below records the exact API without duplicating the implementation.

In [ ]:
RUN_SINGLE_GROUP_EXAMPLE = False
if RUN_SINGLE_GROUP_EXAMPLE:
    single_group_input = build_irf_input(
        accessibility=hvr_accessibility,
        region_names=hvr_factors.index,
        region_by_factor=hvr_factors,
        open_fraction_threshold=0.5,
        entropy_minimum=0.05,
    )
    display(single_group_input.head())

## Checkpoints, outputs, and next step

The canonical path uses 38,659 x 23 and 11,921 x 23 frozen factor matrices and writes 13 deterministic iRF input Parquet files. Raw RIT terminology is introduced in the next step; selected objects become factor-combination modules only after source-specific selection.

**Next step:** switch to the Python 3.6 iRF environment and run `02_irf_rit.py`.